<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/weather_api_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24 requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 96.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


In [ ]:
#This file contains code snippets for creating and using a weather tool with an agent in LangChain
# ── Google Colab API Key Setup ──────────────────────────────────────────────
# Store OPENAI_API_KEY and WEATHER_API_KEY in Colab Secrets (key icon in sidebar).
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL']  = 'https://openai.vocareum.com/v1'


In [ ]:
# To do: create api key and keep in env file as WEATHER_API_KEY from this url: https://www.weatherapi.com/

os.environ['WEATHER_API_KEY'] = userdata.get('WEATHER_API_KEY')

API_KEY = os.environ['WEATHER_API_KEY']

In [ ]:
from langchain.tools import tool
from langchain.agents import AgentExecutor , create_tool_calling_agent

from langchain_openai import ChatOpenAI

import requests


In [ ]:
@tool
def get_weather(city:str) -> str:

    """
    Fetch the currect weather for a given city using Weather API
    """
    url = f"https://api.weatherapi.com/v1/current.json?key={API_KEY}&q={city}"
    data = requests.get(url).json()

    temp = data['current']['temp_c']
    condition = data['current']['condition']['text']

    return f"The weather in {city} : {temp} , {condition}"


tools = [get_weather]

In [ ]:
# LLM + Agent

llm = ChatOpenAI(
    temperature=0,
    model_name='gpt-4o-mini')

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert weather assistant. Use the tools provided to answer weather-related queries accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


In [ ]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template)

executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
query = "What's the weather in London right now?"
result = executor.invoke({"input": query})
print("Agent Output:", result)




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'London'}`


The weather in London : 2.2 , SunnyThe current weather in London is 2.2°C and sunny.

> Finished chain.
Agent Output: {'input': "What's the weather in London right now?", 'output': 'The current weather in London is 2.2°C and sunny.'}
